# Testing the Agentic Knowledge Engineering Tool
This notebook sets up the `Orchestrator` to test the Distillation, Designer, and Validation loops with `instructor` and `openai` against your configured LLM (e.g. Vertex AI or local Ollama).


In [1]:
import sys
import os
# Allow imports from local directory if running inside the ontology folder
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

from core.models import DocumentSource
from pipeline.orchestrator import Orchestrator


## Define Test Documents
Here we define the input raw texts that need to be distilled into a Knowledge Graph and synthesized into a strict Pydantic schema.


In [2]:
doc1 = DocumentSource(
    id="doc_001",
    text_content="""
    Patient Assessment - Oct 4, 2023:
    Jordan exhibited highly repetitive physical motions, specifically hand-flapping, 
    when the classroom environment became too loud. He did not engage in verbal communication for 45 minutes.
    """
)

doc2 = DocumentSource(
    id="doc_002",
    text_content="""
    In-home Observation - Oct 5, 2023:
    During dinner, Jordan successfully maintained eye contact with his sibling when asking for water.
    However, when transitioned to bedtime, severe physical agitation and distress was observed.
    """
)

docs = [doc1, doc2]


## Initialize Orchestrator and Run Pipeline
We point the Orchestrator to the configured LLM endpoint using generic environment variables (`LLM_BASE_URL`). Note: ensure your endpoint is running or correct credentials are provided."


In [3]:
import os
from dotenv import load_dotenv
load_dotenv()
from core.auth import get_api_key

# Force the Google Auth parser to activate 
os.environ["LLM_PROVIDER"] = "google"

orchestrator = Orchestrator(
    model_name=os.getenv("LLM_MODEL_NAME", "mistral-small-agent"), 
    base_url=os.getenv("LLM_BASE_URL", "http://localhost:11434/v1"),
    api_key=get_api_key(),  # get_api_key() will see LLM_PROVIDER="google" and fetch the Token
    hallucination_filter=True,
    ontology_depth=None,
    strict_typing=True,
    verbose=True
)

[AUTH WARNING] Could not fetch Google Auth token: ('invalid_scope: Invalid OAuth scope or ID token audience provided.', {'error': 'invalid_scope', 'error_description': 'Invalid OAuth scope or ID token audience provided.'})
[*] Orchestrator initialized. Model: qwen/qwen3-5@qwen3.5-9b | Base URL: https://us-central1-aiplatform.googleapis.com/v1/projects/knowledgeontology/locations/us-central1/endpoints/mg-endpoint-7217b3a8-8aaa-4312-9bed-9a6dd59cb13b:predict


In [3]:


try:
    final_schema = orchestrator.run_pipeline(docs)
except Exception as e:
    print(f"Pipeline Error: {e}")
    print("Is your LLM connection running locally, or did you export LLM_BASE_URL?")
    final_schema = None


[*] Orchestrator initialized. Model: mistral-small-agent | Base URL: http://localhost:11434/v1
====== I. DISTILLATION & II. DESIGNER ======
[*] Starting Consolidation Loop over 2 documents...


KeyboardInterrupt: 

## Synthesized Schema Output
If the validation passes strictly with zero false positives, we can now view the dynamic schema produced.


In [ ]:
if final_schema:
    print("Final Dynamic Schema JSON Definition:\n")
    import json
    print(json.dumps(final_schema.model_json_schema(), indent=2))
